In [ ]:
## Part 1 — Install Dependencies
"""

# Commented out IPython magic to ensure Python compatibility.
# %%capture
# !pip install fastapi uvicorn[standard] structlog tenacity pybreaker \
#              prometheus-client prometheus-fastapi-instrumentator \
#              httpx pyngrok nest-asyncio

import nest_asyncio
nest_asyncio.apply()  # Allow asyncio inside Colab's event loop
print('✅ Dependencies installed and asyncio patched')

"""

In [1]:
!pip install fastapi uvicorn[standard] structlog tenacity pybreaker \
             prometheus-client prometheus-fastapi-instrumentator \
             httpx pyngrok nest-asyncio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.3/73.3 kB 4.4 MB/s eta 0:00:00
  Attempting uninstall: starlette
    Found existing installation: starlette 0.52.1
    Uninstalling starlette-0.52.1:
      Successfully uninstalled starlette-0.52.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires starlette<1.0.0,>=0.49.1, but you have starlette 1.2.1 which is incompatible.
gradio 5.50.0 requires starlette<1.0,>=0.40.0, but you have starlette 1.2.1 which is incompatible.


In [2]:
import nest_asyncio
nest_asyncio.apply()  # Allow asyncio inside Colab's event loop
print('✅ Dependencies installed and asyncio patched')

✅ Dependencies installed and asyncio patched


In [17]:
import structlog, logging, uuid, time
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse

# Configure structlog to output clean JSON
structlog.configure(
    processors=[
        structlog.stdlib.add_log_level,
        structlog.stdlib.add_logger_name,
        structlog.processors.TimeStamper(fmt='iso'),
        structlog.processors.JSONRenderer()
    ],
    wrapper_class=structlog.BoundLogger,
    logger_factory=structlog.PrintLoggerFactory(),
)

log = structlog.get_logger()

app = FastAPI(title='EY Payment API', version='1.0.0')

@app.middleware('http')
async def logging_middleware(request: Request, call_next):
    correlation_id = request.headers.get('X-Correlation-Id', str(uuid.uuid4()))
    correlation_id_var.set(correlation_id)
    start = time.perf_counter()

    log.info('request.started',
             path=request.url.path,
             method=request.method,
             correlation_id=correlation_id)

    response = await call_next(request)
    elapsed_ms = round((time.perf_counter() - start) * 1000, 2)

    log.info('request.completed',
             path=request.url.path,
             status=response.status_code,
             latency_ms=elapsed_ms,
             correlation_id=correlation_id)

    response.headers['X-Correlation-Id'] = correlation_id
    return response

@app.get('/health/live')
async def liveness():
    return {'status': 'alive'}

@app.get('/health/ready')
async def readiness():
    # In production, check DB + MQ here
    return {'status': 'ready', 'db': 'ok', 'mq': 'ok'}

@app.post('/payments')
async def create_payment(request: Request):
    body = await request.json()
    log.info('payment.received', amount=body.get('amount'), currency=body.get('currency', 'GBP'))
    return {'payment_id': str(uuid.uuid4()), 'status': 'accepted', **body}

print('✅ Logging middleware and routes defined')

✅ Logging middleware and routes defined


In [4]:
from prometheus_client import Counter, Histogram, generate_latest, CONTENT_TYPE_LATEST
from prometheus_fastapi_instrumentator import Instrumentator
from fastapi.responses import Response

# Custom business metric: track payment amounts by currency
PAYMENT_AMOUNT = Histogram(
    'payment_amount_gbp', 'Payment amount in GBP',
    buckets=[10, 50, 100, 500, 1000, 5000, 10000]
)

ERROR_COUNT = Counter(
    'payment_errors_total', 'Total payment processing errors',
    ['error_type']
)

# Auto-instrument all HTTP routes (latency, status codes, method)
Instrumentator().instrument(app).expose(app)

print('✅ Prometheus metrics instrumented')
print('   → /metrics endpoint is now available')
print('   → Tracking: request_count, request_latency, payment_amount')

✅ Prometheus metrics instrumented
   → /metrics endpoint is now available
   → Tracking: request_count, request_latency, payment_amount


In [5]:
import random
import asyncio
from tenacity import (
    retry, stop_after_attempt, wait_exponential,
    retry_if_exception_type, before_sleep_log
)

# Simulate a flaky downstream fraud-check service
call_count = 0

async def flaky_fraud_check(payload: dict) -> dict:
    global call_count
    call_count += 1
    # Fail the first 2 attempts, succeed on 3rd
    if call_count < 3:
        print(f'  ⚠️  Attempt {call_count}: fraud API timeout (simulated)')
        raise ConnectionError(f'Fraud API timeout on attempt {call_count}')
    print(f'  ✅ Attempt {call_count}: fraud check passed')
    return {'fraud_score': 0.02, 'decision': 'approved'}


@retry(
    stop=stop_after_attempt(5),
    wait=wait_exponential(multiplier=1, min=0.1, max=2),  # seconds (small for demo)
    retry=retry_if_exception_type((ConnectionError, TimeoutError)),
    reraise=True
)
async def call_fraud_api_with_retry(payload: dict) -> dict:
    return await flaky_fraud_check(payload)


# Test it
print('🔄 Testing retry logic...')
result = asyncio.run(
    call_fraud_api_with_retry({'amount': 1500, 'currency': 'GBP'})
)
print(f'\n📦 Final result: {result}')

🔄 Testing retry logic...
  ⚠️  Attempt 1: fraud API timeout (simulated)
  ⚠️  Attempt 2: fraud API timeout (simulated)
  ✅ Attempt 3: fraud check passed

📦 Final result: {'fraud_score': 0.02, 'decision': 'approved'}


In [6]:
import pybreaker

class LoggingListener(pybreaker.CircuitBreakerListener):
    def state_change(self, cb, old_state, new_state):
        log.warning('circuit_breaker.state_change',
                    breaker=cb.name,
                    old=str(old_state),
                    new=str(new_state))

fraud_breaker = pybreaker.CircuitBreaker(
    fail_max=3,
    reset_timeout=30,
    listeners=[LoggingListener()],
    name='fraud-api'
)

def check_fraud_cb(payload: dict) -> dict:
    """Wrapped by circuit breaker — raises CircuitBreakerError when OPEN."""
    # Simulate always-failing service for demo
    raise ConnectionError('Fraud service down')

def safe_fraud_check(payload: dict) -> dict:
    """Call fraud API with circuit breaker; return fallback if circuit is OPEN."""
    try:
        return fraud_breaker.call(check_fraud_cb, payload)
    except pybreaker.CircuitBreakerError:
        print('🔴 Circuit OPEN — returning safe fallback (manual review)')
        return {'fraud_score': None, 'decision': 'manual_review', 'circuit': 'open'}
    except Exception as e:
        return {'fraud_score': None, 'decision': 'error', 'reason': str(e)}

print('📊 Simulating calls to a failing fraud service...')
for i in range(6):
    result = safe_fraud_check({'amount': 200 * (i + 1)})
    print(f'  Call {i+1}: {result} | Circuit state: {fraud_breaker.current_state}')


📊 Simulating calls to a failing fraud service...
  Call 1: {'fraud_score': None, 'decision': 'error', 'reason': 'Fraud service down'} | Circuit state: closed
  Call 2: {'fraud_score': None, 'decision': 'error', 'reason': 'Fraud service down'} | Circuit state: closed
  Call 3: {'fraud_score': None, 'decision': 'error', 'reason': "'PrintLogger' object has no attribute 'name'"} | Circuit state: closed
  Call 4: {'fraud_score': None, 'decision': 'error', 'reason': "'PrintLogger' object has no attribute 'name'"} | Circuit state: closed
  Call 5: {'fraud_score': None, 'decision': 'error', 'reason': "'PrintLogger' object has no attribute 'name'"} | Circuit state: closed
  Call 6: {'fraud_score': None, 'decision': 'error', 'reason': "'PrintLogger' object has no attribute 'name'"} | Circuit state: closed


In [7]:
NGROK_TOKEN = 'YOUR_NGROK_TOKEN_HERE'  # ← REPLACE THIS WITH YOUR ACTUAL NGROK TOKEN

import threading, uvicorn
from pyngrok import ngrok, conf

conf.get_default().auth_token = NGROK_TOKEN

def run_server():
    uvicorn.run(app, host='0.0.0.0', port=8000, log_level='warning')

thread = threading.Thread(target=run_server, daemon=True)
thread.start()

import time; time.sleep(2)
tunnel = ngrok.connect(8000)
public_url = tunnel.public_url

print(f'🌐 Public URL: {public_url}')
print(f'   Health:   {public_url}/health/ready')
print(f'   Metrics:  {public_url}/metrics')
print(f'   Docs:     {public_url}/docs')

# Quick smoke tests using httpx
import httpx, json

BASE = public_url  # set in the cell above

with httpx.Client() as client:
    # 1 — Health check
    r = client.get(f'{BASE}/health/ready')
    print('Health:', r.json())

    # 2 — POST a payment (watch structured logs above)
    r = client.post(f'{BASE}/payments',
                    json={'amount': 1500, 'currency': 'GBP', 'account': 'ACC-42'},
                    headers={'X-Correlation-Id': 'demo-corr-001'})
    print('Payment:', r.json())
    print('Returned correlation ID:', r.headers.get('X-Correlation-Id'))

    # 3 — Peek at first 500 chars of /metrics
    r = client.get(f'{BASE}/metrics')
    print('\nMetrics (first 500 chars):')
    print(r.text[:500])

/usr/local/lib/python3.12/dist-packages/uvicorn/server.py:75: RuntimeWarning: coroutine 'Server.serve' was never awaited
  return asyncio_run(self.serve(sockets=sockets), loop_factory=self.config.get_loop_factory())
Exception in thread Thread-3 (run_server):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipykernel_1075/176397580.py", line 9, in run_server
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/main.py", line 620, in run
    server.run()
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/server.py", line 75, in run
    return asyncio_run(self.serve(sockets=sockets), loop_factory=self.config.get_loop_factory())
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: _patch_asyncio.<locals>.run() got an unexpected keyword 

ERROR:pyngrok.process.ngrok:t=2026-06-09T05:04:50+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: The authtoken you specified does not look like a proper ngrok authtoken.\nYour authtoken: PASTE_YOUR_TOKEN_HERE\nInstructions to install your authtoken are on your ngrok dashboard:\nhttps://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_105\r\n"
ERROR:pyngrok.process.ngrok:t=2026-06-09T05:04:50+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: The authtoken you specified does not look like a proper ngrok authtoken.\nYour authtoken: PASTE_YOUR_TOKEN_HERE\nInstructions to install your authtoken are on your ngrok dashboard:\nhttps://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_105\r\n"


PyngrokNgrokError: The ngrok process errored on start: authentication failed: The authtoken you specified does not look like a proper ngrok authtoken.\nYour authtoken: PASTE_YOUR_TOKEN_HERE\nInstructions to install your authtoken are on your ngrok dashboard:\nhttps://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_105\r\n.

🔥 Extension Tasks

In [ ]:
### Extension A — Rate-Limiting Middleware *(Medium)*
# Add a sliding-window rate limiter (e.g., 100 req/min per client IP) using an in-memory dictionary.
# - Return `429 Too Many Requests` with a `Retry-After` header when the limit is exceeded
# - Track limit hits as a Prometheus `Counter`

In [8]:
from prometheus_client import Counter

RATE_LIMIT_HITS = Counter(
    "rate_limit_hits_total",
    "Total requests rejected due to rate limiting"
)

In [9]:
from collections import defaultdict
from datetime import datetime, timedelta

REQUEST_LOG = defaultdict(list)

RATE_LIMIT = 100
WINDOW_SECONDS = 60

In [16]:
from fastapi.responses import JSONResponse

@app.middleware("http")
async def rate_limit_middleware(request: Request, call_next):

    client_ip = request.client.host
    now = datetime.utcnow()

    REQUEST_LOG[client_ip] = [
        ts for ts in REQUEST_LOG[client_ip]
        if now - ts < timedelta(seconds=WINDOW_SECONDS)
    ]

    if len(REQUEST_LOG[client_ip]) >= RATE_LIMIT:

        RATE_LIMIT_HITS.inc()

        return JSONResponse(
            status_code=429,
            headers={"Retry-After": "60"},
            content={
                "error": "Rate limit exceeded"
            }
        )

    REQUEST_LOG[client_ip].append(now)

    return await call_next(request)

In [ ]:
### Extension B — Correlation ID Propagation *(Medium)*
# The current middleware assigns a correlation ID but does NOT forward it to outbound `httpx` calls.
# - Add a `contextvars.ContextVar` to store the correlation ID per request
# - Write a custom `httpx.Auth` or event hook that injects `X-Correlation-Id` into every outbound call
# - Verify it appears in mock downstream requests

In [14]:
import contextvars
correlation_id_var = contextvars.ContextVar(
    "correlation_id",
    default=None
)

In [18]:
correlation_id = correlation_id_var.get()

correlation_id_var.set(correlation_id)

<Token var=<ContextVar name='correlation_id' default=None at 0x7f8a1f21cf40> at 0x7f8a1f0b2e00>

In [19]:
import httpx

async def add_correlation_id(request):

    correlation_id = correlation_id_var.get()

    if correlation_id:
        request.headers["X-Correlation-Id"] = correlation_id

In [20]:
client = httpx.AsyncClient(
    event_hooks={
        "request": [add_correlation_id]
    }
)

In [ ]:
# # Validation
# response = await client.get(
#     "https://api.example.com/fraud-check"
# )

In [ ]:
# ### Extension C — Prometheus + Grafana Dashboard *(Advanced)*
# - Export your running `/metrics` to a free [Grafana Cloud](https://grafana.com/products/cloud/) account using the Prometheus remote-write endpoint
# - Build a dashboard with 4 panels: request rate, p99 latency, error rate, circuit breaker trip count

In [23]:
# Create Counter:
CIRCUIT_BREAKER_TRIPS = Counter(
    "circuit_breaker_trips_total",
    "Number of breaker openings"
)

In [24]:
# Increment when breaker opens:
def state_change(self, cb, old, new):

    if str(new) == "open":
        CIRCUIT_BREAKER_TRIPS.inc()

In [25]:
def state_change(self, cb, old, new):

    if str(new) == "open":
        CIRCUIT_BREAKER_TRIPS.inc()

In [ ]:
# ### Extension D — Structured Log Aggregation *(Advanced)*
# - Set up a free [Elastic Cloud](https://www.elastic.co/cloud/) cluster or use the Colab local filesystem
# - Configure `structlog` to write logs as newline-delimited JSON to a file
# - Write a query that finds all requests with `latency_ms > 200` for a specific `correlation_id`

In [26]:
import logging

logging.basicConfig(
    filename="payments.log",
    level=logging.INFO
)

In [29]:
# Configure Structlog
structlog.configure(
    processors=[
        structlog.processors.TimeStamper(fmt="iso"),
        structlog.processors.JSONRenderer()
    ]
)

In [ ]:
# Query Slow Requests
import json

with open("payments.log") as f:  # Validate once payments.log file is created

    for line in f:

        record = json.loads(line)

        if (
            record.get("latency_ms", 0) > 200
            and record.get("correlation_id")
                == "abc123"
        ):
            print(record)